# Social Mobility Indicator Analysis

## Project Overview
Analyze socioeconomic and educational indicator data to explore patterns linked to upward mobility outcomes across regions.

## Learning Objectives
- Explore composite social mobility indicators
- Correlate education, income, and opportunity metrics
- Identify regional clusters of high and low mobility
- Analyze intergenerational income patterns

## Problem Statement
Policymakers need to understand where and why upward mobility occurs — or fails to occur — so they can design targeted interventions to expand economic opportunity.

## Why This Project Matters
Social mobility is a core measure of economic fairness. Data-driven analysis reveals structural barriers and highlights regions that succeed in enabling upward movement.

## Dataset Overview
Synthetic regional dataset: 200 counties with education attainment, median income, poverty rate, mobility score, and demographic indicators.

## Dataset Source and License Notes
- Synthetic data inspired by Opportunity Insights / Census-style indicators
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 200
region = np.random.choice(['Urban', 'Suburban', 'Rural'], n, p=[0.30, 0.35, 0.35])
state_cluster = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West'], n)

# Core indicators
college_rate = np.clip(np.random.normal(35, 12, n), 8, 75).round(1)
hs_grad_rate = np.clip(np.random.normal(85, 8, n), 55, 99).round(1)
median_income = np.clip(np.random.normal(55000, 18000, n), 20000, 130000).astype(int)
poverty_rate = np.clip(100 - college_rate * 0.5 - median_income / 2000 + np.random.normal(0, 5, n), 3, 40).round(1)
unemployment = np.clip(np.random.normal(5.5, 2, n), 1.5, 15).round(1)
single_parent_pct = np.clip(np.random.normal(30, 10, n), 8, 60).round(1)
segregation_idx = np.clip(np.random.beta(2, 3, n) * 100, 5, 85).round(1)

# Mobility score: composite driven by education + income - poverty - segregation
mobility_score = np.clip(
    college_rate * 0.3 + hs_grad_rate * 0.15 + median_income / 3000
    - poverty_rate * 0.4 - single_parent_pct * 0.15 - segregation_idx * 0.1
    + np.random.normal(0, 3, n), 5, 60
).round(1)

# Child income rank (percentile of child's income given parent at 25th percentile)
child_income_rank = np.clip(mobility_score + 15 + np.random.normal(0, 4, n), 20, 70).round(1)

df = pd.DataFrame({
    'CountyID': [f'C{i:04d}' for i in range(n)],
    'Region': region, 'StateCluster': state_cluster,
    'CollegeRate': college_rate, 'HSGradRate': hs_grad_rate,
    'MedianIncome': median_income, 'PovertyRate': poverty_rate,
    'Unemployment': unemployment, 'SingleParentPct': single_parent_pct,
    'SegregationIdx': segregation_idx,
    'MobilityScore': mobility_score, 'ChildIncomeRank': child_income_rank
})
print(f'Dataset: {df.shape}')
print(f'Mobility score range: {df["MobilityScore"].min()} — {df["MobilityScore"].max()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nKey stats:')
for col in ['MobilityScore', 'ChildIncomeRank', 'CollegeRate', 'PovertyRate']:
    print(f'  {col}: mean={df[col].mean():.1f}, std={df[col].std():.1f}')

## Mobility Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['MobilityScore'].hist(bins=25, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Mobility Score Distribution')
axes[0].axvline(df['MobilityScore'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["MobilityScore"].mean():.1f}')
axes[0].legend()
sns.boxplot(data=df, x='Region', y='MobilityScore', ax=axes[1])
axes[1].set_title('Mobility Score by Region Type')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'mobility_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation Matrix

In [ ]:
corr_cols = ['CollegeRate', 'HSGradRate', 'MedianIncome', 'PovertyRate',
             'Unemployment', 'SingleParentPct', 'SegregationIdx', 'MobilityScore', 'ChildIncomeRank']
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Indicator Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

## Education and Mobility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['CollegeRate', 'HSGradRate'], ['College Rate', 'HS Graduation Rate']):
    ax.scatter(df[col], df['MobilityScore'], alpha=0.4, s=20)
    z = np.polyfit(df[col], df['MobilityScore'], 1)
    ax.plot(np.sort(df[col]), np.polyval(z, np.sort(df[col])), 'r-', lw=2)
    r = df[col].corr(df['MobilityScore'])
    ax.set_title(f'{title} vs Mobility (r={r:.2f})')
    ax.set_xlabel(title + ' (%)')
    ax.set_ylabel('Mobility Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'education_mobility.png'), dpi=100, bbox_inches='tight')
plt.show()

## Poverty and Segregation Effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, color in zip(axes, ['PovertyRate', 'SegregationIdx'], ['coral', 'purple']):
    ax.scatter(df[col], df['ChildIncomeRank'], alpha=0.4, s=20, color=color)
    z = np.polyfit(df[col], df['ChildIncomeRank'], 1)
    ax.plot(np.sort(df[col]), np.polyval(z, np.sort(df[col])), 'r-', lw=2)
    r = df[col].corr(df['ChildIncomeRank'])
    ax.set_title(f'{col} vs Child Income Rank (r={r:.2f})')
    ax.set_xlabel(col)
    ax.set_ylabel('Child Income Rank (percentile)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'poverty_segregation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Cluster Comparison

In [ ]:
cluster_stats = df.groupby('StateCluster')[['MobilityScore', 'ChildIncomeRank', 'CollegeRate',
                                                    'PovertyRate', 'SegregationIdx']].mean().round(1)
print('State Cluster Averages:')
print(cluster_stats)

fig, ax = plt.subplots(figsize=(10, 6))
cluster_stats[['MobilityScore', 'ChildIncomeRank']].plot.bar(ax=ax, edgecolor='black')
ax.set_title('Mobility Metrics by State Cluster')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cluster_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Top and Bottom Mobility Counties

In [ ]:
print('Top 10 Mobility Counties:')
top10 = df.nlargest(10, 'MobilityScore')[['CountyID', 'Region', 'StateCluster', 'MobilityScore',
                                            'CollegeRate', 'PovertyRate', 'MedianIncome']]
print(top10.to_string(index=False))
print('\nBottom 10 Mobility Counties:')
bot10 = df.nsmallest(10, 'MobilityScore')[['CountyID', 'Region', 'StateCluster', 'MobilityScore',
                                             'CollegeRate', 'PovertyRate', 'MedianIncome']]
print(bot10.to_string(index=False))

## Interpretation and Business Insight
- **College attainment** is the strongest positive correlate of upward mobility
- **Poverty rate** and **segregation** are the strongest negative correlates
- **Single-parent household rate** shows a meaningful negative association with mobility — reflecting structural challenges, not individual blame
- **Urban areas** show higher average mobility scores, driven by education and income access
- **Regional clusters** reveal geographic patterns — some areas consistently enable or hinder mobility
- The strongest levers for policy are education access and poverty reduction

## Limitations
- Synthetic data with simplified causal structure
- No individual-level tracking (ecological fallacy risk)
- No temporal dynamics (single snapshot)
- No race/ethnicity disaggregation
- Mobility score is a composite — real mobility measurement is complex

## How to Improve This Project
- Add longitudinal mobility tracking (parent → child income over 20+ years)
- Include race/ethnicity disaggregation
- Build causal models (instrumental variables, quasi-experiments)
- Add policy intervention data (e.g., school funding changes, housing programs)
- Link to health and criminal justice outcomes

## Production Considerations
- Regional mobility dashboards for policymakers
- Opportunity atlas-style interactive maps
- Automated disparity reporting for equity audits
- Integration with census and tax record data

## Common Mistakes
- Confusing correlation with causation in observational indicator data
- Drawing individual conclusions from aggregate (county-level) data
- Ignoring that mobility metrics depend heavily on how income is measured
- Treating mobility as a single-number summary when it varies by race, gender, and starting point

## Mini Challenge / Exercises
1. Which indicator has the strongest correlation with ChildIncomeRank?
2. Among Rural counties, what is the average mobility score vs Suburban?
3. Find counties with high college rate (> 50%) but low mobility score (< 25). What explains this?
4. If poverty rate decreased by 5 percentage points, estimate the expected mobility score change using the linear relationship.

## Final Summary / Key Takeaways
- Social mobility is driven by education access, economic opportunity, and community structure
- Poverty and segregation are the strongest barriers to upward mobility
- Geographic disparities are substantial — where you grow up matters
- Effective policy targets multiple indicators simultaneously
- Data analysis can identify high-leverage interventions but cannot replace robust causal research